# 16장. 허깅페이스 공개

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


[Hugging Face](https://huggingface.co)는 AI 모델을 공유하는 가장 큰 저장소입니다. 모델을 올리면 다른 사람이 URL 하나로 받아 쓸 수 있습니다.

> **공개는 되돌리기 어렵습니다.** 한 번 공개한 파일은 누군가 이미 받아 갔을 수 있고, 삭제해도 저장소 기록(git 이력)에 남습니다. 이 장의 점검을 **반드시** 거친 뒤 올리세요. 이 노트북의 셀은 준비와 점검만 하고, 실제 업로드는 하지 않습니다.

## 16.1 무엇을 올리나

| 올릴 것 | 이유 |
|---|---|
| `onnx/student.onnx` + `.onnx.data` (14장 정리본) | 모델 |
| `verify/pi_infer.py`, `verify/pi_verify_data.npz` | 받은 사람이 결과를 재현할 수 있게 |
| `README.md` (모델 카드) | 모델 설명서 |

학습 체크포인트(`.pt`)는 선택입니다. Teacher `.pt`는 458 MB이고 추론에는 필요 없습니다.

## 16.2 모델 카드 (README.md)

모델 카드 맨 위의 `---` 사이 부분(YAML)은 허깅페이스가 읽는 **메타데이터**입니다.

```yaml
---
license: cc-by-nc-4.0          # 라이선스
library_name: onnx             # 어떤 도구로 실행하나
base_model: AutonLab/MOMENT-1-small
tags: [onnx, time-series-classification, bearing-fault-diagnosis, raspberry-pi]
---
```

본문에는 입력 형식(전처리 계약), 파일 설명, 출처를 적습니다.

## 16.3 라이선스: 데이터 조건을 따른다

Paderborn 데이터는 **CC BY-NC 4.0**(출처 표기, **비상업적** 이용)입니다. 이 데이터로 학습한 모델도 이 조건을 따르는 것이 안전하므로 `cc-by-nc-4.0`으로 표기합니다. 기반 모델 MOMENT의 라이선스도 각 모델 카드에서 확인하세요.

In [2]:
import os, warnings
warnings.filterwarnings("ignore")
import ch16_huggingface as ch16
print(ch16.MODEL_CARD.format(title="(제목)")[:400], "...")

---
license: cc-by-nc-4.0
library_name: onnx
base_model: AutonLab/MOMENT-1-small
tags:
  - onnx
  - time-series-classification
  - bearing-fault-diagnosis
  - raspberry-pi
---

# (제목)

> **Experimental model.** A vibration anomaly analysis model that runs on Raspberry Pi.
> Additional training may be required depending on the application environment.

## Files
- `onnx/student.onnx`, `onnx/student. ...


## 16.4 업로드 폴더 준비와 공개 전 점검

점검 항목:
- 컴퓨터 경로(`/Users/이름`, `/home/이름`, `C:\Users`): 14장의 ONNX stack trace, 절대 경로가 찍힌 로그나 JSON
- 허깅페이스 토큰(`hf_...`), 개인 키
- 노트북 출력, 설정 파일, 검증 결과 JSON에 남은 경로

In [3]:
%run code/ch16_huggingface.py

업로드 폴더: outputs/manual/hf_upload
  README.md                       0.0 MB
  onnx/student.onnx               0.5 MB
  onnx/student.onnx.data        602.2 MB
  verify/pi_infer.py              0.0 MB
  verify/pi_verify_data.npz       0.9 MB



공개 전 점검: 문제 없음


정리하지 **않은** ONNX(13장 원본)를 넣으면 점검이 잡아내는지 확인합니다. 결과는 개수만 출력하고, 실제 경로 문자열은 출력하지 않습니다.

In [4]:
import shutil, tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    shutil.copy2(ROOT / "outputs/manual/onnx/student.onnx", tmp / "student.onnx")
    for path, name, n in ch16.scan(tmp):
        print(f"⚠ {path}: {name} {n}건 → 이대로 올리면 사용자 이름이 공개된다")

⚠ student.onnx: macOS 사용자 경로 2465건 → 이대로 올리면 사용자 이름이 공개된다


## 16.5 업로드 (터미널에서, 본인이 직접)

1. 허깅페이스 계정을 만들고, **Settings → Access Tokens**에서 **write** 권한 토큰을 만듭니다.
2. 터미널에서 로그인합니다. 토큰은 화면에 붙여 넣기만 하고, **코드나 노트북에 적지 마세요.**
   ```bash
   hf auth login
   hf auth whoami          # 로그인 확인
   ```
3. 점검을 통과하면 업로드합니다. 처음에는 `--private`(비공개)로 올려 확인한 뒤 공개로 바꾸는 것을 권장합니다.
   ```bash
   python docs/manual/code/ch16_huggingface.py --upload --repo-id 내계정/저장소이름 --private
   ```

## 16.6 이미 올린 것을 고치고 싶을 때

| 상황 | 방법 | 주의 |
|---|---|---|
| 파일을 새 버전으로 바꾸기 | 같은 저장소에 다시 업로드 | 옛 파일은 **커밋 기록에 남는다** |
| 기록까지 정리 | `HfApi().super_squash_history(repo_id)`로 커밋을 하나로 합치기 | 되돌릴 수 없음. 옛 커밋 번호를 아는 사람은 한동안 접근 가능할 수 있음 |
| 완전 삭제 | 웹의 저장소 Settings에서 삭제 후 재생성 | 다운로드 수, 좋아요 등도 사라짐 |

이 프로젝트는 실제로 사용자 경로가 든 ONNX를 한 번 공개했고, 새 버전으로 덮어쓴 뒤 기록을 합쳤습니다. 그래도 옛 커밋 번호로는 한동안 접근할 수 있었습니다. **처음부터 점검하고 올리는 것**이 유일하게 확실한 방법입니다.

## 16장 완성 코드: `code/ch16_huggingface.py`

In [5]:
from IPython.display import Code
Code(filename=str(CODE / "ch16_huggingface.py"), language="python")

"""16장 완성 코드: 허깅페이스 업로드 폴더를 준비하고, 공개 전 점검을 한다. (업로드는 --upload 를 줄 때만)

    1) 업로드 폴더 만들기: onnx/, verify/, README.md(모델 카드)
    2) 공개 전 점검: 파일 안에 컴퓨터 경로·사용자 이름·토큰처럼 보이는 문자열이 없는지
    3) (선택) 업로드: 먼저 터미널에서 `hf auth login` 으로 로그인해 두어야 한다

실행 (프로젝트 루트에서, 14·15장 결과 사용):
    python docs/manual/code/ch16_huggingface.py                         # 준비 + 점검만
    python docs/manual/code/ch16_huggingface.py --upload --repo-id 내계정/저장소이름 --private
"""

import argparse
import re
import shutil
from pathlib import Path

ROOT = Path(__file__).resolve().parents[3]

MODEL_CARD = """---
license: cc-by-nc-4.0
library_name: onnx
base_model: AutonLab/MOMENT-1-small
tags:
  - onnx
  - time-series-classification
  - bearing-fault-diagnosis
  - raspberry-pi
---

# {title}

> **Experimental model.** A vibration anomaly analysis model that runs on Raspberry Pi.
> Additional training may be required depending on the application environment.

## Files
- `onnx/student.onnx`, `onnx/student.onnx.data` — keep both in the same folder, do not rename.
- `verify/pi_infer.py`, `verify/pi_verify_data.npz` — parity check: `python verify/pi_infer.py verify`

## Input
Single-axis vibration (64 kHz) → 1 kHz low-pass (8th-order Butterworth, zero-phase) → 4 kHz →
1-s windows (4000 samples, stride 2000) → 8 chunks × 512 (last 96 zero-padded, mask 0).
ONNX input `x_enc [1,1,512]`, `input_mask [1,512]` → `logits [1,2]`. Window decision = mean of 8 chunk logits.

## Attribution
Data: Paderborn University Bearing DataCenter (CC BY-NC 4.0). Base model: AutonLab/MOMENT-1-small.
"""

# 공개하면 안 되는 것처럼 보이는 문자열 패턴
PATTERNS = {
    "macOS 사용자 경로": re.compile(rb"/Users/[^/\s\"']+"),
    "Linux 사용자 경로": re.compile(rb"/home/[^/\s\"']+"),
    "Windows 사용자 경로": re.compile(rb"C:\\\\Users\\\\"),
    "허깅페이스 토큰": re.compile(rb"hf_[A-Za-z0-9]{30,}"),
    "개인 키": re.compile(rb"-----BEGIN [A-Z ]*PRIVATE KEY-----"),
}


def prepare(onnx_path, bundle_dir, out_dir, title):
    if out_dir.exists():
        shutil.rmtree(out_dir)
    (out_dir / "onnx").mkdir(parents=True)
    (out_dir / "verify").mkdir()
    for f in (onnx_path, Path(str(onnx_path) + ".data")):
        shutil.copy2(f, out_dir / "onnx" / f.name)
    # 15장 스크립트는 같은 폴더의 student.onnx를 찾으므로, 업로드 구조(onnx/ 폴더)에 맞게 경로만 바꾼다
    script = (bundle_dir / "pi_infer.py").read_text().replace('HERE / "student.onnx"', 'HERE.parent / "onnx" / "student.onnx"')
    (out_dir / "verify" / "pi_infer.py").write_text(script)
    shutil.copy2(bundle_dir / "pi_verify_data.npz", out_dir / "verify" / "pi_verify_data.npz")
    (out_dir / "README.md").write_text(MODEL_CARD.format(title=title))


def scan(folder):
    """모든 파일을 바이트 단위로 훑어 위험한 문자열을 찾는다 (.onnx 안의 메타데이터 포함)."""
    findings = []
    for f in sorted(folder.rglob("*")):
        if f.is_file():
            data = f.read_bytes()
            for name, pattern in PATTERNS.items():
                hits = pattern.findall(data)
                if hits:
                    findings.append((f.relative_to(folder), name, len(hits)))
    return findings


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--onnx", default="outputs/manual/onnx_clean/student.onnx")
    parser.add_argument("--bundle", default="outputs/manual/pi_bundle")
    parser.add_argument("--output-dir", default="outputs/manual/hf_upload")
    parser.add_argument("--title", default="MOMENT-Small Bearing Vibration Classifier (ONNX, manual build)")
    parser.add_argument("--upload", action="store_true")
    parser.add_argument("--repo-id")
    parser.add_argument("--private", action="store_true")
    args = parser.parse_args()

    out = ROOT / args.output_dir
    prepare(ROOT / args.onnx, ROOT / args.bundle, out, args.title)
    print(f"업로드 폴더: {out.relative_to(ROOT)}")
    for f in sorted(out.rglob("*")):
        if f.is_file():
            print(f"  {str(f.relative_to(out)):28s} {f.stat().st_size / 1e6:6.1f} MB")

    findings = scan(out)
    print("\n공개 전 점검:", "문제 없음" if not findings else "")
    for path, name, n in findin